# eval-multi (local) — IFEval base vs N SFT checkpoints

Локальная версия. Читает все `ckpt-step-*.pt` из подпапки `./checkpoints/` рядом с этим ноутбуком и сравнивает их с baseline `Qwen/Qwen3-1.7B-Base` через IFEval.

## Подготовка

```bash
# Положи ckpt в подпапку:
mkdir -p checkpoints
cp /path/to/ckpt-step-2000.pt checkpoints/
cp /path/to/ckpt-step-3500.pt checkpoints/

# Зависимости:
pip install torch transformers lm-eval immutabledict langdetect nltk absl-py bitsandbytes
```

## Параметры

- `MODEL_NAME = Qwen/Qwen3-1.7B-Base`
- `NUM_BENCHS = 100` — кол-во IFEval промптов (None = все 541)
- `SHUFFLE_SEED = 42` — те же индексы для всех моделей
- `do_sample = False` (greedy), `max_new_tokens = 1024`

## Hardware

Одна GPU (cuda:0). На multi-GPU машине ограничивается через `CUDA_VISIBLE_DEVICES=0`. Каждая модель выгружается из VRAM до загрузки следующей.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # одна GPU

import gc, json, random, shutil
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"
NUM_BENCHS = 100
SHUFFLE_SEED = 42

# Локальные пути относительно ноутбука
NOTEBOOK_DIR = Path.cwd()
CKPT_DIR = NOTEBOOK_DIR / "checkpoints"
OUT_DIR = NOTEBOOK_DIR / "eval_out"
OUT_DIR.mkdir(exist_ok=True)

CHATML_TEMPLATE = (
    "{% for m in messages %}"
    "<|im_start|>{{ m['role'] }}\n{{ m['content'] }}<|im_end|>\n"
    "{% endfor %}"
    "{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}"
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: CPU — IFEval будет очень медленный (~часы на модель)")

In [ ]:
"""Auto-discovery всех ckpt-step-*.pt в локальной checkpoints/."""

all_ckpts = sorted(
    CKPT_DIR.glob("ckpt-step-*.pt"),
    key=lambda p: int(p.stem.rsplit("-", 1)[-1]),
)
print(f"discovered {len(all_ckpts)} ckpt(s) in {CKPT_DIR}:")
for p in all_ckpts:
    print(f"  step={int(p.stem.rsplit('-',1)[-1])}  {p.name}  ({p.stat().st_size/1e9:.2f} GB)")

if not all_ckpts:
    print(f"\nWARNING: ни одного ckpt-step-*.pt не найдено в {CKPT_DIR}")
    print("Будет eval только base (для контроля что pipeline работает).")

In [ ]:
def _load_tokenizer():
    tok = AutoTokenizer.from_pretrained(
        MODEL_NAME, chat_template=CHATML_TEMPLATE, padding_side="left",
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def load_base_model():
    tok = _load_tokenizer()
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, attn_implementation="sdpa",
    ).to(DEVICE).eval()
    return model, tok


def load_sft_from_ckpt(ckpt_path):
    """FSDP FULL_STATE_DICT payload["model"] (fp32) → HF arch (fp16 для inference).
    Аккуратно освобождаем RAM: payload (~6.8 GB fp32) и sd (~3.4 GB fp16) идут в del+gc
    как только их данные впитались в model. Иначе пик RAM = ~13.6 GB на одну загрузку.
    """
    print(f"[SFT] loading {ckpt_path.name}")
    tok = _load_tokenizer()

    payload = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    sd = {k: v.to(torch.float16) for k, v in payload["model"].items()}
    step = payload.get("step", "?")
    del payload                                  # ~6.8 GB fp32 RAM освобождаем
    gc.collect()

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, attn_implementation="sdpa",
    )
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing or unexpected:
        print(f"  state_dict — missing: {len(missing)}, unexpected: {len(unexpected)}")

    del sd                                       # ~3.4 GB fp16 RAM освобождаем
    gc.collect()

    model = model.to(DEVICE).eval()
    return model, tok, step


def _free_gpu(*objs):
    """Освобождает GPU/RAM от переданных объектов (модель, tokenizer и т.д.).
    Вызывать как `_free_gpu(model, tok)` ИМЕННО на caller-side scope. del внутри
    функции освобождает только локальные ссылки, поэтому тут просто принудительный
    gc + empty_cache, а сами объекты caller должен `del` отдельно."""
    gc.collect()
    gc.collect()                                  # 2-й проход — gc cycles из HFLM
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def _export_to_hf(model, tok, out_dir):
    out_dir = Path(out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)
    return out_dir

In [ ]:
_SELECTED_INDICES = None

def _compute_selected_indices(total):
    global _SELECTED_INDICES
    if _SELECTED_INDICES is not None:
        return _SELECTED_INDICES
    if NUM_BENCHS is None:
        _SELECTED_INDICES = list(range(total))
    else:
        rng = random.Random(SHUFFLE_SEED)
        idxs = list(range(total))
        rng.shuffle(idxs)
        _SELECTED_INDICES = sorted(idxs[:NUM_BENCHS])
    return _SELECTED_INDICES


def _run_lm_eval(hf_dir, out_dir, batch_size=4, max_new_tokens=1024, seed=42):
    """Monkey-patch get_task_dict — одни и те же индексы IFEval для всех моделей."""
    import lm_eval
    from lm_eval import evaluator as lm_evaluator

    out_dir = Path(out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    def _patch_task_dict(td):
        for k, v in td.items():
            if isinstance(v, dict):
                _patch_task_dict(v)
            elif hasattr(v, "dataset") and v.dataset is not None:
                ds = v.dataset
                split = "train" if "train" in ds else list(ds.keys())[0]
                indices = _compute_selected_indices(len(ds[split]))
                ds[split] = ds[split].select(indices)
                print(f"[lm_eval] {k}: {len(indices)}/541 (seed={SHUFFLE_SEED})")

    original = lm_evaluator.get_task_dict
    def _patched(*args, **kwargs):
        td = original(*args, **kwargs)
        _patch_task_dict(td)
        return td
    lm_evaluator.get_task_dict = _patched
    try:
        results = lm_evaluator.simple_evaluate(
            model="hf",
            model_args=f"pretrained={hf_dir},dtype=float16,device=cuda:0" if DEVICE == "cuda" else f"pretrained={hf_dir},dtype=float32",
            tasks=["ifeval"],
            batch_size=batch_size,
            apply_chat_template=True,
            gen_kwargs=f"do_sample=False,max_new_tokens={max_new_tokens}",
            log_samples=False,
            random_seed=seed,
            numpy_random_seed=seed,
            torch_random_seed=seed,
        )
    finally:
        lm_evaluator.get_task_dict = original

    (out_dir / "results.json").write_text(
        json.dumps(results["results"], indent=2, default=str, ensure_ascii=False)
    )

    metrics = results["results"]["ifeval"]
    del results
    gc.collect()
    gc.collect()                              # 2-й проход — HFLM может оставить cycles
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    return metrics

In [ ]:
@torch.no_grad()
def _chat_once(model, tok, history, max_new_tokens=128):
    prompt = tok.apply_chat_template(history, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(DEVICE)
    out = model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def chat_sanity(model, tok, label, user_messages, max_new_tokens=128):
    print(f"\n=== chat sanity — {label} ===")
    history, out_history = [], []
    for msg in user_messages:
        history.append({"role": "user", "content": msg})
        print(f"\n[USER] {msg}")
        resp = _chat_once(model, tok, history, max_new_tokens=max_new_tokens)
        history.append({"role": "assistant", "content": resp})
        print(f"[ASSISTANT] {resp}")
        out_history.append({"user": msg, "assistant": resp})
    return out_history

In [ ]:
def pretty_ifeval(results):
    """results — list[{label, metrics}]. Pretty-table с Δ vs base."""
    keys = [
        ("prompt_level_strict_acc,none", "prompt-level strict"),
        ("prompt_level_loose_acc,none",  "prompt-level loose"),
        ("inst_level_strict_acc,none",   "instr-level strict"),
        ("inst_level_loose_acc,none",    "instr-level loose"),
    ]

    by_label = {r["label"]: r["metrics"] for r in results}
    base_label = next((l for l in by_label if l == "base"), None)
    sft_labels = [l for l in by_label if l != "base"]
    cols = ([base_label] if base_label else []) + sft_labels

    name_w, col_w, delta_w = 24, 14, 12
    show_delta = base_label is not None
    n_d = len(sft_labels) if show_delta else 0
    sep_total = name_w + col_w * len(cols) + delta_w * n_d
    bar = "═" * sep_total

    print()
    print(bar)
    print("  IFEval comparison")
    print(bar)

    header = "metric".ljust(name_w)
    for c in cols:
        header += c.rjust(col_w)
    if show_delta:
        for s in sft_labels:
            header += f"Δ{s[:9]}".rjust(delta_w)
    print(header)
    print("─" * sep_total)

    for k_full, k_short in keys:
        row = k_short.ljust(name_w)
        vals = {}
        for c in cols:
            v = by_label[c].get(k_full)
            if isinstance(v, (int, float)):
                vals[c] = float(v)
                row += f"{vals[c]*100:>{col_w-1}.2f}%"
            else:
                row += f"{'-':>{col_w}}"
        if show_delta:
            base_v = vals.get(base_label)
            for s in sft_labels:
                sv = vals.get(s)
                if base_v is not None and sv is not None:
                    d = (sv - base_v) * 100
                    row += f"{d:+.2f}%".rjust(delta_w)
                else:
                    row += f"{'-':>{delta_w}}"
        print(row)
    print(bar)

    (OUT_DIR / "metrics_ifeval_compare_multi.json").write_text(
        json.dumps(results, indent=2, ensure_ascii=False)
    )
    print(f"\nsaved → {OUT_DIR / 'metrics_ifeval_compare_multi.json'}")

In [ ]:
def main():
    user_messages = [
        "What is the capital of France? Answer with exactly one word and nothing else.",
        "Now translate your previous answer to Spanish.",
    ]

    chats, metrics = [], []

    # 1) Base
    print("\n" + "="*60 + "\n  BASE: Qwen/Qwen3-1.7B-Base\n" + "="*60)
    bm, bt = load_base_model()
    chats.append({"label": "base", "history": chat_sanity(bm, bt, "base", user_messages)})
    base_hf = _export_to_hf(bm, bt, OUT_DIR / "hf_base")
    del bm, bt                                   # ВАЖНО: только del в caller-scope
    _free_gpu()                                  # реально освобождает после удаления ссылок
    print("\nrunning IFEval on base...")
    metrics.append({"label": "base", "metrics": _run_lm_eval(base_hf, OUT_DIR / "ifeval_base")})
    shutil.rmtree(base_hf, ignore_errors=True)
    _free_gpu()

    # 2) Каждый ckpt
    for p in all_ckpts:
        step = int(p.stem.rsplit("-", 1)[-1])
        label = f"sft@{step}"
        print("\n" + "="*60 + f"\n  SFT: step={step}  ({p.name})\n" + "="*60)
        sm, st, _ = load_sft_from_ckpt(p)
        chats.append({"label": label, "history": chat_sanity(sm, st, label, user_messages)})
        hf_dir = _export_to_hf(sm, st, OUT_DIR / f"hf_{label.replace('@','_')}")
        del sm, st                               # ВАЖНО: только del в caller-scope
        _free_gpu()
        print(f"\nrunning IFEval on {label}...")
        metrics.append({"label": label, "metrics": _run_lm_eval(hf_dir, OUT_DIR / f"ifeval_{label.replace('@','_')}")})
        shutil.rmtree(hf_dir, ignore_errors=True)
        _free_gpu()

    # 3) Pretty
    pretty_ifeval(metrics)

    print("\n" + "="*60 + "\n  Chat dialogues\n" + "="*60)
    for c in chats:
        print(f"\n### {c['label']}")
        for turn in c["history"]:
            print(f"  [USER] {turn['user']}")
            print(f"  [ASST] {turn['assistant']}")


if __name__ == "__main__":
    main()